In [ ]:
import subprocess
try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('no GPU (nvidia-smi not found)')

In [ ]:
import pathlib, os, sys, shutil

for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.count(os.sep) - '/kaggle/input'.count(os.sep)
    if depth <= 3:
        print(root)
    if depth >= 4:
        dirs[:] = []

matches = [p.parent.parent for p in pathlib.Path('/kaggle/input').rglob('src/config.py')
           if (p.parent.parent / 'scripts' / 'train_sft.py').exists() and '/project/' not in str(p)]
assert matches, 'repo dataset (src/config.py + scripts/) not found under /kaggle/input'
repo_src = matches[0]
print('repo dataset found at', repo_src)

proj = pathlib.Path('/kaggle/working/project')
if not proj.exists():
    shutil.copytree(repo_src, proj)
os.chdir(proj)
sys.path.insert(0, str(proj))
os.environ['PYTHONPATH'] = str(proj)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import subprocess
def run(cmd):
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed with exit code {proc.returncode}: {cmd}')

import importlib
for name in ('torch', 'transformers', 'datasets', 'peft', 'trl'):
    try:
        print(name, importlib.import_module(name).__version__)
    except Exception as error:
        print(name, 'unavailable:', error)

## Confirm kaggle_benchmarks is importable (requires 'Set as Benchmark Task')

In [ ]:
import kaggle_benchmarks
print('kaggle_benchmarks module found:', kaggle_benchmarks)
print([n for n in dir(kaggle_benchmarks) if not n.startswith('_')])

## Locate pairwise files from kernel_sources

In [ ]:
import glob
hh_pairs = glob.glob('/kaggle/input/**/project/outputs/eval/dpo_hh_rlhf_vs_sft_pairs.jsonl', recursive=True)
uf_pairs = glob.glob('/kaggle/input/**/project/outputs/eval/dpo_ultrafeedback_vs_sft_pairs.jsonl', recursive=True)
assert hh_pairs and uf_pairs, (hh_pairs, uf_pairs)
hh_pairs_path = hh_pairs[0]
uf_pairs_path = uf_pairs[0]
print('hh_pairs_path', hh_pairs_path)
print('uf_pairs_path', uf_pairs_path)

## Task 7 -- judge HH-RLHF track (safety_mode on: safety-sensitive eval set)

In [ ]:
run(f"python scripts/judge_pairwise_kaggle.py --pairs_path {hh_pairs_path} --output_path outputs/eval/dpo_hh_rlhf_vs_sft_judged.jsonl --judge_model google/gemini-3.6-flash --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments --safety_mode")

## Task 7 -- judge UltraFeedback/AlpacaEval track

In [ ]:
run(f"python scripts/judge_pairwise_kaggle.py --pairs_path {uf_pairs_path} --output_path outputs/eval/dpo_ultrafeedback_vs_sft_judged.jsonl --judge_model google/gemini-3.6-flash --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments")

## Confirm outputs

In [ ]:
import pathlib
for p in sorted(pathlib.Path('outputs/eval').rglob('*judged*')):
    print(p)
for p in sorted(pathlib.Path('outputs/eval').rglob('*failures*')):
    print(p)